# NuevaMente — Retrieval V2 Ablation Study

## Objetivo

Este notebook mide por separado el aporte de:

1. Retrieval V2 sin metadata filtering.
2. Metadata prefilter por `document_id`.

La comparación mantiene constantes:

- Ground Truth v2.
- Corpus `chunks_v1.csv`.
- 50 queries originales.
- `top_k = 5`.
- embedding model.
- búsqueda vectorial.
- BM25.
- Reciprocal Rank Fusion (RRF).
- overfetch x3.
- Cross-Encoder reranking.

## Condiciones evaluadas

| Condición | Descripción |
|---|---|
| V1 baseline | Retrieval baseline evaluado con Ground Truth v2 |
| V2 unfiltered | Retrieval híbrido + RRF + Cross-Encoder, sin metadata filter |
| V2 filtered | Pipeline V2 completo con prefilter por `document_id` |

## Métricas

- Recall@3
- Recall@5
- Precision@3
- Precision@5

La ablación permite distinguir cuánto de la mejora corresponde al nuevo motor de retrieval y cuánto se añade al incorporar metadata filtering.

In [1]:
from pathlib import Path
import json
import hashlib
import math

import pandas as pd

In [2]:
# ============================================================
# RAÍZ DEL PROYECTO DATA_IA
# ============================================================

PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
EVALUATION_DIR = DATA_DIR / "evaluation"
INPUT_DIR = EVALUATION_DIR / "input"
RESULTS_DIR = EVALUATION_DIR / "results"

ABLATION_OUTPUT_DIR = RESULTS_DIR / "retrieval_v2_ablation"
ABLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUND_TRUTH_PATH = EVALUATION_DIR / "ground_truth_v2.csv"
CHUNKS_PATH = EVALUATION_DIR / "chunks_v1.csv"

BASELINE_METRICS_PATH = RESULTS_DIR / "retrieval_metrics_gt_v2.csv"
BASELINE_RESULTS_PATH = RESULTS_DIR / "retrieval_results_gt_v2.csv"

V2_FILTERED_PATH = INPUT_DIR / "retrieval_results_agentes_v2.json"
V2_UNFILTERED_PATH = INPUT_DIR / "retrieval_results_agentes_v2_unfiltered.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT:", ABLATION_OUTPUT_DIR)

PROJECT_ROOT: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA
OUTPUT: E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2_ablation


In [3]:
required_paths = [
    GROUND_TRUTH_PATH,
    CHUNKS_PATH,
    BASELINE_METRICS_PATH,
    BASELINE_RESULTS_PATH,
    V2_FILTERED_PATH,
    V2_UNFILTERED_PATH,
]

missing = []

for path in required_paths:
    exists = path.exists()
    print("✅" if exists else "❌", path)

    if not exists:
        missing.append(path)

assert not missing, (
    "Faltan archivos requeridos:\n"
    + "\n".join(str(p) for p in missing)
)

✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\ground_truth_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\chunks_v1.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_metrics_gt_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_results_gt_v2.csv
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\input\retrieval_results_agentes_v2.json
✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\input\retrieval_results_agentes_v2_unfiltered.json


In [4]:
ground_truth_df = pd.read_csv(
    GROUND_TRUTH_PATH,
    encoding="utf-8-sig",
)

assert len(ground_truth_df) == 50
assert ground_truth_df["case_id"].nunique() == 50
assert ground_truth_df["pregunta"].notna().all()
assert ground_truth_df["relevant_chunk_ids"].notna().all()

print("✅ Ground Truth v2 válido")
print("Casos:", len(ground_truth_df))
print("Categorías:", sorted(ground_truth_df["categoria"].unique()))

ground_truth_df[
    [
        "case_id",
        "document_id",
        "categoria",
        "pregunta",
        "relevant_chunk_ids",
    ]
].head()

✅ Ground Truth v2 válido
Casos: 50
Categorías: ['Backend', 'Cloud / DevOps', 'Data Science', 'Frontend', 'IA']


,case_id,document_id,categoria,pregunta,relevant_chunk_ids
0,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,AI-ES-001_CH_001
1,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,AI-ES-001_CH_001
2,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,AI-ES-001_CH_002
3,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,AI-ES-001_CH_002
4,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,AI-ES-001_CH_001


In [5]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8-sig") as f:
        return json.load(f)


def parse_chunk_ids(value):
    if value is None:
        return []

    if isinstance(value, float) and math.isnan(value):
        return []

    if isinstance(value, list):
        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]

    text = str(value).strip()

    if not text:
        return []

    if ";" in text:
        return [
            x.strip()
            for x in text.split(";")
            if x.strip()
        ]

    return [text]


def recall_at_k(relevant, retrieved, k):
    relevant_set = set(relevant)

    if not relevant_set:
        return 0.0

    return (
        len(relevant_set.intersection(retrieved[:k]))
        / len(relevant_set)
    )


def precision_at_k(relevant, retrieved, k):
    if k <= 0:
        return 0.0

    return (
        len(set(relevant).intersection(retrieved[:k]))
        / k
    )


def sha256_file(path: Path):
    hasher = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            hasher.update(block)

    return hasher.hexdigest()

In [6]:
v2_filtered_batch = load_json(V2_FILTERED_PATH)
v2_unfiltered_batch = load_json(V2_UNFILTERED_PATH)

print("V2 filtered:", len(v2_filtered_batch))
print("V2 unfiltered:", len(v2_unfiltered_batch))

assert len(v2_filtered_batch) == 50
assert len(v2_unfiltered_batch) == 50

V2 filtered: 50
V2 unfiltered: 50


In [7]:
gt_queries = (
    ground_truth_df
    .set_index("case_id")["pregunta"]
    .to_dict()
)

gt_case_ids = set(gt_queries)


def validate_batch(batch, expected_filter_mode):
    case_ids = [case["case_id"] for case in batch]

    assert len(case_ids) == 50
    assert len(set(case_ids)) == 50
    assert set(case_ids) == gt_case_ids

    for case in batch:
        assert case["contract_version"] == "2.0"
        assert case["top_k"] == 5
        assert case["score_type"] == "cross_encoder"
        assert case["status"] == "success"

        assert (
            case["query"].strip()
            == gt_queries[case["case_id"]].strip()
        )

        assert (
            case.get("filter_mode")
            == expected_filter_mode
        )


validate_batch(
    v2_filtered_batch,
    expected_filter_mode="document_id",
)

validate_batch(
    v2_unfiltered_batch,
    expected_filter_mode="none",
)

print("✅ V2 filtered validado")
print("✅ V2 unfiltered validado")
print("✅ 50/50 queries idénticas al benchmark")
print("✅ Contrato 2.0")
print("✅ top_k = 5")
print("✅ score_type = cross_encoder")

✅ V2 filtered validado
✅ V2 unfiltered validado
✅ 50/50 queries idénticas al benchmark
✅ Contrato 2.0
✅ top_k = 5
✅ score_type = cross_encoder


In [8]:
gt_by_case = (
    ground_truth_df
    .set_index("case_id")
    .to_dict("index")
)


def evaluate_batch(batch, condition):
    rows = []

    for case in batch:
        case_id = case["case_id"]
        gt = gt_by_case[case_id]

        relevant = parse_chunk_ids(
            gt["relevant_chunk_ids"]
        )

        retrieved = [
            result["chunk_id"]
            for result in case.get("results", [])
            if result.get("chunk_id")
        ]

        retrieved_documents = [
            result.get("document_id")
            for result in case.get("results", [])
        ]

        rows.append(
            {
                "condition": condition,
                "case_id": case_id,
                "document_id": gt["document_id"],
                "categoria": gt["categoria"],
                "pregunta": gt["pregunta"],
                "relevant_chunk_ids": relevant,
                "retrieved_chunk_ids": retrieved,
                "retrieved_document_ids": retrieved_documents,
                "recall_at_3": recall_at_k(
                    relevant, retrieved, 3
                ),
                "recall_at_5": recall_at_k(
                    relevant, retrieved, 5
                ),
                "precision_at_3": precision_at_k(
                    relevant, retrieved, 3
                ),
                "precision_at_5": precision_at_k(
                    relevant, retrieved, 5
                ),
            }
        )

    return pd.DataFrame(rows)

In [9]:
filtered_df = evaluate_batch(
    v2_filtered_batch,
    "V2 filtered",
)

unfiltered_df = evaluate_batch(
    v2_unfiltered_batch,
    "V2 unfiltered",
)

print("Filtered:", len(filtered_df))
print("Unfiltered:", len(unfiltered_df))

filtered_df.head()

Filtered: 50
Unfiltered: 50


,condition,case_id,document_id,categoria,pregunta,relevant_chunk_ids,retrieved_chunk_ids,retrieved_document_ids,recall_at_3,recall_at_5,precision_at_3,precision_at_5
0,V2 filtered,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]","[AI-ES-001, AI-ES-001]",1.0,1.0,0.333333,0.2
1,V2 filtered,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]","[AI-ES-001, AI-ES-001]",1.0,1.0,0.333333,0.2
2,V2 filtered,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,[AI-ES-001_CH_002],"[AI-ES-001_CH_002, AI-ES-001_CH_001]","[AI-ES-001, AI-ES-001]",1.0,1.0,0.333333,0.2
3,V2 filtered,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,[AI-ES-001_CH_002],"[AI-ES-001_CH_002, AI-ES-001_CH_001]","[AI-ES-001, AI-ES-001]",1.0,1.0,0.333333,0.2
4,V2 filtered,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-001_CH_002]","[AI-ES-001, AI-ES-001]",1.0,1.0,0.333333,0.2


In [10]:
baseline_metrics_df = pd.read_csv(
    BASELINE_METRICS_PATH,
    encoding="utf-8-sig",
)

baseline_lookup = (
    baseline_metrics_df
    .set_index("metric")["mean"]
    .to_dict()
)

baseline = {
    "Recall@3": float(
        baseline_lookup["recall_at_3"]
    ),
    "Recall@5": float(
        baseline_lookup["recall_at_5"]
    ),
    "Precision@3": float(
        baseline_lookup["precision_at_3"]
    ),
    "Precision@5": float(
        baseline_lookup["precision_at_5"]
    ),
}

baseline

{'Recall@3': 0.5733333333333333,
 'Recall@5': 0.7533333333333333,
 'Precision@3': 0.2133333333333333,
 'Precision@5': 0.1639999999999999}

In [11]:
def summarize(df):
    return {
        "Recall@3": df["recall_at_3"].mean(),
        "Recall@5": df["recall_at_5"].mean(),
        "Precision@3": df["precision_at_3"].mean(),
        "Precision@5": df["precision_at_5"].mean(),
    }


unfiltered_summary = summarize(unfiltered_df)
filtered_summary = summarize(filtered_df)

global_comparison_df = pd.DataFrame(
    {
        "V1 baseline": baseline,
        "V2 unfiltered": unfiltered_summary,
        "V2 filtered": filtered_summary,
    }
)

global_comparison_df[
    "delta_unfiltered_vs_v1"
] = (
    global_comparison_df["V2 unfiltered"]
    - global_comparison_df["V1 baseline"]
)

global_comparison_df[
    "delta_filter"
] = (
    global_comparison_df["V2 filtered"]
    - global_comparison_df["V2 unfiltered"]
)

global_comparison_df[
    "delta_total_v2_vs_v1"
] = (
    global_comparison_df["V2 filtered"]
    - global_comparison_df["V1 baseline"]
)

global_comparison_df.round(6)

,V1 baseline,V2 unfiltered,V2 filtered,delta_unfiltered_vs_v1,delta_filter,delta_total_v2_vs_v1
Recall@3,0.573333,0.813333,0.933333,0.240000,0.12,0.360000
Recall@5,0.753333,0.853333,0.953333,0.100000,0.10,0.200000
Precision@3,0.213333,0.300000,0.340000,0.086667,0.04,0.126667
Precision@5,0.164000,0.188000,0.208000,0.024000,0.02,0.044000


## Interpretación global

Los resultados permiten separar dos efectos:

### Aporte del nuevo Retrieval V2 sin filtros

V2 unfiltered mejora al baseline V1 incluso sin metadata filtering.

Esto indica que la combinación de:

- búsqueda vectorial;
- BM25;
- RRF;
- overfetching;
- Cross-Encoder reranking;

aporta una mejora independiente del filtro por documento.

### Aporte adicional del metadata prefilter

Al incorporar `document_id` se observa una mejora adicional tanto en Recall como en Precision.

Por lo tanto, el rendimiento final de Retrieval V2 no depende exclusivamente del metadata filtering: ambos componentes contribuyen al resultado.

In [12]:
case_comparison_df = (
    unfiltered_df[
        [
            "case_id",
            "document_id",
            "categoria",
            "pregunta",
            "relevant_chunk_ids",
            "retrieved_chunk_ids",
            "recall_at_3",
            "recall_at_5",
            "precision_at_3",
            "precision_at_5",
        ]
    ]
    .merge(
        filtered_df[
            [
                "case_id",
                "retrieved_chunk_ids",
                "recall_at_3",
                "recall_at_5",
                "precision_at_3",
                "precision_at_5",
            ]
        ],
        on="case_id",
        suffixes=("_unfiltered", "_filtered"),
    )
)

case_comparison_df[
    "delta_recall_at_3_filter"
] = (
    case_comparison_df["recall_at_3_filtered"]
    - case_comparison_df["recall_at_3_unfiltered"]
)

case_comparison_df[
    "delta_recall_at_5_filter"
] = (
    case_comparison_df["recall_at_5_filtered"]
    - case_comparison_df["recall_at_5_unfiltered"]
)

case_comparison_df[
    "delta_precision_at_3_filter"
] = (
    case_comparison_df["precision_at_3_filtered"]
    - case_comparison_df["precision_at_3_unfiltered"]
)

case_comparison_df[
    "delta_precision_at_5_filter"
] = (
    case_comparison_df["precision_at_5_filtered"]
    - case_comparison_df["precision_at_5_unfiltered"]
)

case_comparison_df.head()

,case_id,document_id,categoria,pregunta,relevant_chunk_ids,retrieved_chunk_ids_unfiltered,recall_at_3_unfiltered,recall_at_5_unfiltered,precision_at_3_unfiltered,precision_at_5_unfiltered,retrieved_chunk_ids_filtered,recall_at_3_filtered,recall_at_5_filtered,precision_at_3_filtered,precision_at_5_filtered,delta_recall_at_3_filter,delta_recall_at_5_filter,delta_precision_at_3_filter,delta_precision_at_5_filter
0,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-002_CH_001, AI-ES-002...",1.0,1.0,0.333333,0.2,"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,0.333333,0.2,0.0,0.0,0.000000,0.0
1,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-002_CH_002, DS-ES-001...",1.0,1.0,0.333333,0.2,"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,0.333333,0.2,0.0,0.0,0.000000,0.0
2,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,[AI-ES-001_CH_002],"[AI-ES-001_CH_002, CLD-ES-001_CH_006, CLD-ES-0...",1.0,1.0,0.333333,0.2,"[AI-ES-001_CH_002, AI-ES-001_CH_001]",1.0,1.0,0.333333,0.2,0.0,0.0,0.000000,0.0
3,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,[AI-ES-001_CH_002],"[CLD-ES-002_CH_004, FE-ES-001_CH_035, CLD-ES-0...",0.0,1.0,0.000000,0.2,"[AI-ES-001_CH_002, AI-ES-001_CH_001]",1.0,1.0,0.333333,0.2,1.0,0.0,0.333333,0.0
4,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,[AI-ES-001_CH_001],"[AI-ES-001_CH_001, AI-ES-002_CH_002, FE-ES-002...",1.0,1.0,0.333333,0.2,"[AI-ES-001_CH_001, AI-ES-001_CH_002]",1.0,1.0,0.333333,0.2,0.0,0.0,0.000000,0.0


In [13]:
rescued_by_filter_df = case_comparison_df[
    (
        case_comparison_df["recall_at_5_unfiltered"] == 0
    )
    &
    (
        case_comparison_df["recall_at_5_filtered"] > 0
    )
].copy()

print(
    "Casos rescatados por document_id:",
    len(rescued_by_filter_df),
)

rescued_by_filter_df[
    [
        "case_id",
        "document_id",
        "categoria",
        "pregunta",
        "recall_at_5_unfiltered",
        "recall_at_5_filtered",
        "retrieved_chunk_ids_unfiltered",
        "retrieved_chunk_ids_filtered",
    ]
]

Casos rescatados por document_id: 5


,case_id,document_id,categoria,pregunta,recall_at_5_unfiltered,recall_at_5_filtered,retrieved_chunk_ids_unfiltered,retrieved_chunk_ids_filtered
7,AI-ES-002-Q03,AI-ES-002,IA,¿Qué habilidad relacionada con prompts se espe...,0.0,1.0,"[CLD-ES-001_CH_014, FE-ES-002_CH_006, FE-ES-00...","[AI-ES-002_CH_002, AI-ES-002_CH_001]"
23,DS-ES-001-Q04,DS-ES-001,Data Science,¿Qué ejercicio práctico aparece en el módulo?,0.0,1.0,"[DS-ES-002_CH_002, AI-ES-001_CH_001, AI-ES-001...","[DS-ES-001_CH_001, DS-ES-001_CH_002]"
29,DS-ES-002-Q05,DS-ES-002,Data Science,¿Qué requisito previo se indica?,0.0,1.0,"[CLD-ES-002_CH_003, FE-ES-002_CH_002, AI-ES-00...","[DS-ES-002_CH_001, DS-ES-002_CH_002]"
36,FE-ES-002-Q02,FE-ES-002,Frontend,¿Qué conocimientos previos se recomiendan?,0.0,1.0,"[CLD-ES-002_CH_003, AI-ES-002_CH_002, AI-ES-00...","[FE-ES-002_CH_007, FE-ES-002_CH_002, FE-ES-002..."
39,FE-ES-002-Q05,FE-ES-002,Frontend,¿El módulo se limita únicamente a sintaxis bás...,0.0,1.0,"[FE-ES-002_CH_004, AI-ES-001_CH_001, DS-ES-002...","[FE-ES-002_CH_004, FE-ES-002_CH_002, FE-ES-002..."


In [14]:
remaining_failures_df = case_comparison_df[
    case_comparison_df["recall_at_5_filtered"] == 0
].copy()

print(
    "Fallos restantes V2 filtered:",
    len(remaining_failures_df),
)

remaining_failures_df[
    [
        "case_id",
        "document_id",
        "categoria",
        "pregunta",
        "relevant_chunk_ids",
        "retrieved_chunk_ids_filtered",
        "recall_at_3_filtered",
        "recall_at_5_filtered",
    ]
]

Fallos restantes V2 filtered: 2


,case_id,document_id,categoria,pregunta,relevant_chunk_ids,retrieved_chunk_ids_filtered,recall_at_3_filtered,recall_at_5_filtered
17,BE-ES-002-Q03,BE-ES-002,Backend,¿Qué comando crea el proyecto mysite dentro de...,[BE-ES-002_CH_005],"[BE-ES-002_CH_006, BE-ES-002_CH_010, BE-ES-002...",0.0,0.0
44,CLD-ES-001-Q05,CLD-ES-001,Cloud / DevOps,Menciona dos formas de describir a Kubernetes ...,[CLD-ES-001_CH_002],"[CLD-ES-001_CH_003, CLD-ES-001_CH_014, CLD-ES-...",0.0,0.0


In [15]:
filter_regressions_df = case_comparison_df[
    (
        case_comparison_df[
            "delta_recall_at_5_filter"
        ] < 0
    )
    |
    (
        case_comparison_df[
            "delta_precision_at_5_filter"
        ] < 0
    )
].copy()

print(
    "Casos con regresión por filtro:",
    len(filter_regressions_df),
)

filter_regressions_df[
    [
        "case_id",
        "categoria",
        "recall_at_5_unfiltered",
        "recall_at_5_filtered",
        "precision_at_5_unfiltered",
        "precision_at_5_filtered",
    ]
]

Casos con regresión por filtro: 0


,case_id,categoria,recall_at_5_unfiltered,recall_at_5_filtered,precision_at_5_unfiltered,precision_at_5_filtered


In [16]:
metric_columns = [
    "recall_at_3",
    "recall_at_5",
    "precision_at_3",
    "precision_at_5",
]

unfiltered_category_df = (
    unfiltered_df
    .groupby("categoria")[metric_columns]
    .mean()
    .reset_index()
)

filtered_category_df = (
    filtered_df
    .groupby("categoria")[metric_columns]
    .mean()
    .reset_index()
)

category_comparison_df = (
    unfiltered_category_df
    .merge(
        filtered_category_df,
        on="categoria",
        suffixes=("_unfiltered", "_filtered"),
    )
)

for metric in metric_columns:
    category_comparison_df[
        f"delta_{metric}_filter"
    ] = (
        category_comparison_df[
            f"{metric}_filtered"
        ]
        - category_comparison_df[
            f"{metric}_unfiltered"
        ]
    )

category_comparison_df.round(6)

,categoria,recall_at_3_unfiltered,recall_at_5_unfiltered,precision_at_3_unfiltered,precision_at_5_unfiltered,recall_at_3_filtered,recall_at_5_filtered,precision_at_3_filtered,precision_at_5_filtered,delta_recall_at_3_filter,delta_recall_at_5_filter,delta_precision_at_3_filter,delta_precision_at_5_filter
0,Backend,0.866667,0.866667,0.366667,0.22,0.866667,0.866667,0.366667,0.22,0.0,0.0,0.000000,0.00
1,Cloud / DevOps,0.900000,0.900000,0.300000,0.18,0.900000,0.900000,0.300000,0.18,0.0,0.0,0.000000,0.00
2,Data Science,0.800000,0.800000,0.266667,0.16,1.000000,1.000000,0.333333,0.20,0.2,0.2,0.066667,0.04
3,Frontend,0.700000,0.800000,0.266667,0.18,0.900000,1.000000,0.333333,0.22,0.2,0.2,0.066667,0.04
4,IA,0.800000,0.900000,0.300000,0.20,1.000000,1.000000,0.366667,0.22,0.2,0.1,0.066667,0.02


In [17]:
def count_wrong_document_results(row):
    expected_document = row["document_id"]

    return sum(
        1
        for doc_id in row["retrieved_document_ids"]
        if doc_id != expected_document
    )


unfiltered_df[
    "wrong_document_results"
] = unfiltered_df.apply(
    count_wrong_document_results,
    axis=1,
)

print(
    "Promedio de resultados de otros documentos en Top-5:",
    round(
        unfiltered_df[
            "wrong_document_results"
        ].mean(),
        3,
    ),
)

unfiltered_df[
    [
        "case_id",
        "document_id",
        "categoria",
        "wrong_document_results",
        "retrieved_document_ids",
    ]
].sort_values(
    "wrong_document_results",
    ascending=False,
).head(15)

Promedio de resultados de otros documentos en Top-5: 2.22


,case_id,document_id,categoria,wrong_document_results,retrieved_document_ids
7,AI-ES-002-Q03,AI-ES-002,IA,5,"[CLD-ES-001, FE-ES-002, FE-ES-002, AI-ES-001, ..."
23,DS-ES-001-Q04,DS-ES-001,Data Science,5,"[DS-ES-002, AI-ES-001, AI-ES-001, FE-ES-002, A..."
29,DS-ES-002-Q05,DS-ES-002,Data Science,5,"[CLD-ES-002, FE-ES-002, AI-ES-002, BE-ES-002, ..."
2,AI-ES-001-Q03,AI-ES-001,IA,4,"[AI-ES-001, CLD-ES-001, CLD-ES-001, CLD-ES-002..."
8,AI-ES-002-Q04,AI-ES-002,IA,4,"[CLD-ES-002, AI-ES-001, AI-ES-002, FE-ES-002, ..."
3,AI-ES-001-Q04,AI-ES-001,IA,4,"[CLD-ES-002, FE-ES-001, CLD-ES-002, AI-ES-001,..."
6,AI-ES-002-Q02,AI-ES-002,IA,4,"[FE-ES-001, FE-ES-002, AI-ES-002, FE-ES-001, D..."
22,DS-ES-001-Q03,DS-ES-001,Data Science,4,"[DS-ES-002, DS-ES-001, BE-ES-001, CLD-ES-002, ..."
4,AI-ES-001-Q05,AI-ES-001,IA,4,"[AI-ES-001, AI-ES-002, FE-ES-002, CLD-ES-001, ..."
38,FE-ES-002-Q04,FE-ES-002,Frontend,4,"[FE-ES-002, FE-ES-001, FE-ES-001, FE-ES-001, F..."


In [18]:
failure_summary_df = pd.DataFrame(
    [
        {
            "condition": "V2 unfiltered",
            "recall_at_5_zero_cases": int(
                (
                    unfiltered_df["recall_at_5"]
                    == 0
                ).sum()
            ),
        },
        {
            "condition": "V2 filtered",
            "recall_at_5_zero_cases": int(
                (
                    filtered_df["recall_at_5"]
                    == 0
                ).sum()
            ),
        },
    ]
)

failure_summary_df

,condition,recall_at_5_zero_cases
0,V2 unfiltered,7
1,V2 filtered,2


In [19]:
reproducibility_df = pd.DataFrame(
    [
        {
            "artifact": "ground_truth_v2.csv",
            "sha256": sha256_file(
                GROUND_TRUTH_PATH
            ),
        },
        {
            "artifact": "chunks_v1.csv",
            "sha256": sha256_file(
                CHUNKS_PATH
            ),
        },
        {
            "artifact": "retrieval_results_agentes_v2.json",
            "sha256": sha256_file(
                V2_FILTERED_PATH
            ),
        },
        {
            "artifact": "retrieval_results_agentes_v2_unfiltered.json",
            "sha256": sha256_file(
                V2_UNFILTERED_PATH
            ),
        },
    ]
)

reproducibility_df

,artifact,sha256
0,ground_truth_v2.csv,8cd975bc8b295d4a106470be55a65cfc10d3c25a61606c...
1,chunks_v1.csv,309167963f1d5255374f3cb71b5533cfdcd97f02c7fd96...
2,retrieval_results_agentes_v2.json,12c096acc16a4c0be9886637b6040eb74e8f23d1090436...
3,retrieval_results_agentes_v2_unfiltered.json,67f7218be530c71cd0043dab9d1d8d755365e16e60b23d...


In [20]:
GLOBAL_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_global_metrics.csv"
)

CATEGORY_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_category_metrics.csv"
)

CASE_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_case_comparison.csv"
)

RESCUED_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_filter_rescued_cases.csv"
)

FAILURES_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_remaining_failures.csv"
)

REPRO_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_reproducibility.csv"
)


global_comparison_df.to_csv(
    GLOBAL_OUTPUT,
    encoding="utf-8-sig",
)

category_comparison_df.to_csv(
    CATEGORY_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

case_comparison_df.to_csv(
    CASE_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

rescued_by_filter_df.to_csv(
    RESCUED_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

remaining_failures_df.to_csv(
    FAILURES_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

reproducibility_df.to_csv(
    REPRO_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

print("✅ Resultados exportados")

✅ Resultados exportados


In [21]:
ablation_manifest = {
    "experiment": "retrieval_v2_metadata_filter_ablation",
    "ground_truth": "ground_truth_v2.csv",
    "corpus": "chunks_v1.csv",
    "total_cases": 50,
    "top_k": 5,
    "queries_frozen": True,
    "conditions": {
        "v1_baseline": {
            "description": (
                "Retrieval V1 evaluado con GT v2"
            )
        },
        "v2_unfiltered": {
            "metadata_filter": None,
            "vector_search": True,
            "bm25": True,
            "rrf": True,
            "rrf_k": 60,
            "overfetch_factor": 3,
            "cross_encoder": (
                "cross-encoder/"
                "ms-marco-MiniLM-L-6-v2"
            ),
        },
        "v2_filtered": {
            "metadata_filter": "document_id",
            "vector_search": True,
            "bm25": True,
            "rrf": True,
            "rrf_k": 60,
            "overfetch_factor": 3,
            "cross_encoder": (
                "cross-encoder/"
                "ms-marco-MiniLM-L-6-v2"
            ),
        },
    },
    "metrics": [
        "recall_at_3",
        "recall_at_5",
        "precision_at_3",
        "precision_at_5",
    ],
    "hashes": {
        row["artifact"]: row["sha256"]
        for _, row in reproducibility_df.iterrows()
    },
}

MANIFEST_OUTPUT = (
    ABLATION_OUTPUT_DIR
    / "ablation_manifest.json"
)

with MANIFEST_OUTPUT.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        ablation_manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("✅", MANIFEST_OUTPUT)

✅ E:\G10-LATAM-EQUIPO-7-NUEVAMENTE\Data_IA\data\evaluation\results\retrieval_v2_ablation\ablation_manifest.json


# Conclusiones

La ablación muestra que Retrieval V2 mejora al baseline incluso sin metadata filtering.

## Resultado principal

El pipeline V2 sin filtros incrementa:

- Recall@3 de 0.5733 a 0.8133.
- Recall@5 de 0.7533 a 0.8533.
- Precision@3 de 0.2133 a 0.3000.
- Precision@5 de 0.1640 a 0.1880.

Esto demuestra que el nuevo retrieval híbrido + RRF + Cross-Encoder aporta valor independientemente del prefiltrado.

## Impacto de `document_id`

Al añadir metadata prefilter:

- Recall@3 aumenta de 0.8133 a 0.9333.
- Recall@5 aumenta de 0.8533 a 0.9533.
- Precision@3 aumenta de 0.3000 a 0.3400.
- Precision@5 aumenta de 0.1880 a 0.2080.

Los casos con Recall@5 = 0 disminuyen de 7 a 2.

## Conclusión

La mejora total de Retrieval V2 es acumulativa:

1. El motor híbrido y el reranker mejoran sustancialmente el baseline.
2. El metadata prefilter reduce competencia entre documentos y añade una mejora adicional.
3. El rendimiento final no debe atribuirse exclusivamente a uno de los componentes.

El siguiente análisis recomendado es revisar manualmente los dos casos que permanecen con Recall@5 = 0.

## Ablación del reranker

Se compararon las siguientes estrategias sobre los mismos 50 casos, manteniendo fijo el Ground Truth v2, el corpus, `top_k=5`, el filtro por `document_id`, BM25 y RRF:

- RRF-only
- Cross-Encoder
- Fusión RRF + Cross-Encoder con alpha = 0.25
- Fusión RRF + Cross-Encoder con alpha = 0.50
- Fusión RRF + Cross-Encoder con alpha = 0.75

### Resultado

El Cross-Encoder obtuvo el mejor desempeño global en ranking temprano:

- Recall@3 = 0.9333
- Recall@5 = 0.9533
- Precision@3 = 0.3400
- Precision@5 = 0.2080

Las fusiones lineales mantuvieron Recall@5 = 0.9533, pero redujeron Recall@3 respecto al Cross-Encoder puro.

El mejor resultado de fusión fue alpha = 0.50:

- Recall@3 = 0.9133
- Recall@5 = 0.9533
- Precision@3 = 0.3333
- Precision@5 = 0.2080

### Conclusión

La fusión lineal de scores normalizados entre RRF y Cross-Encoder no produjo una mejora sobre el reranking absoluto por Cross-Encoder.

Por lo tanto, se mantiene el Cross-Encoder como estrategia final de reranking para Retrieval V2.

La ablación también mostró que los dos fallos restantes tienen causas diferentes:

- `BE-ES-002-Q03`: el chunk relevante llega correctamente hasta RRF, pero el Cross-Encoder lo degrada fuera del Top-5.
- `CLD-ES-001-Q05`: el chunk relevante es rank 1 en búsqueda vectorial, pero BM25 y RRF reducen su posición y el Cross-Encoder lo degrada aún más.

Esto motiva una ablación adicional de las señales de recuperación iniciales: Vector-only, BM25-only, RRF y RRF + Cross-Encoder.

## Ablación de señales de retrieval

Se compararon las señales Vector-only, BM25-only, RRF y RRF + Cross-Encoder manteniendo constantes el Ground Truth v2, corpus, filtro por `document_id` y `top_k=5`.

| Variante | Recall@3 | Recall@5 | Precision@3 | Precision@5 |
|---|---:|---:|---:|---:|
| Vector-only | 0.7833 | 0.9033 | 0.2867 | 0.1960 |
| BM25-only | 0.5033 | 0.5733 | 0.1867 | 0.1280 |
| RRF | 0.8833 | 0.9533 | 0.3200 | 0.2080 |
| RRF + Cross-Encoder | 0.9333 | 0.9533 | 0.3400 | 0.2080 |

### Conclusión

BM25 presenta un rendimiento bajo como recuperador independiente, pero aporta información complementaria a la búsqueda vectorial.

La fusión mediante RRF mejora Recall@5 de 0.9033 a 0.9533 respecto a Vector-only, demostrando que ambas señales son útiles.

Sin embargo, la fusión RRF con pesos iguales puede degradar consultas donde la señal semántica es especialmente fuerte. En `CLD-ES-001-Q05`, el chunk relevante es recuperado por Vector-only pero se pierde después de incorporar BM25 mediante RRF.

Por lo tanto, no se recomienda eliminar BM25. El siguiente experimento será evaluar Weighted RRF, dando mayor peso a la señal vectorial que a BM25.

## Ablación de Weighted RRF

Se evaluaron diferentes pesos entre la señal vectorial y BM25:

- 50/50
- 60/40
- 70/30
- 80/20

### Resultado

La configuración 60/40 obtuvo el mejor desempeño en Recall@5:

- Recall@3 = 0.8433
- Recall@5 = 0.9800
- Precision@3 = 0.3067
- Precision@5 = 0.2160

También redujo los casos con Recall@5 = 0 de 2 a 1.

El caso `CLD-ES-001-Q05`, que fallaba con RRF 50/50, fue recuperado por RRF 60/40.

Sin embargo, el incremento del peso vectorial reduce Recall@3 respecto al RRF 50/50.

### Conclusión

RRF 60/40 mejora la cobertura en Top-5, pero presenta una ligera degradación en ranking temprano.

Por lo tanto, el siguiente experimento será evaluar RRF 60/40 seguido de Cross-Encoder, para determinar si es posible combinar la mayor cobertura del Weighted RRF con la mejor capacidad de ranking temprano del reranker.

## Weighted RRF + Cross-Encoder

Se comparó el pipeline actual RRF 50/50 + Cross-Encoder contra una variante RRF 60/40 + Cross-Encoder.

### Resultado

| Variante | Recall@3 | Recall@5 | Precision@3 | Precision@5 |
|---|---:|---:|---:|---:|
| RRF 50/50 + Cross-Encoder | 0.9333 | 0.9533 | 0.3400 | 0.2080 |
| RRF 60/40 + Cross-Encoder | 0.9133 | 0.9333 | 0.3333 | 0.2040 |

RRF 60/40 + Cross-Encoder aumentó los casos con Recall@5 = 0 de 2 a 3.

En particular, `BE-ES-001-Q02`, que es recuperado por el pipeline actual, dejó de recuperarse al aplicar pesos 60/40 antes del reranking.

### Conclusión

Aunque RRF 60/40 mejora Recall@5 cuando se evalúa sin reranker, esa ventaja no se conserva al aplicar Cross-Encoder.

Por lo tanto, se mantiene RRF 50/50 + Cross-Encoder como configuración recomendada para Retrieval V2.